<img src="https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/docs/assets/0.-BC-dev-hub-LOGO-flicker.svg" alt="BrainChip Dev Hub" width="200"/>

# Visual Wake Words (VWW) — Akida 2 Benchmark

This notebook evaluates a converted Akida VWW model and, **if an Akida 2 device is connected**, benchmarks its latency. The Akida 2 reference platform is an FPGA running at 25 MHz; a projected latency at a higher clock is also reported (cycle count is clock-independent, so the projection is exact). Power measurement is not yet available, so benchmarking is latency-only.

> **Note:** the hardware benchmark section requires a physical Akida 2 FPGA board. Without a device it will report "no hardware found" and skip — it cannot run on Colab.

## Setup

In [7]:
import os
import numpy as np
import akida

from brainchip_utils.hardware_utils import AKIDA_CLOCKS_HZ

DATA_PATH = './data/vw_coco2014_96'
MODELS_DIR = './models'
INPUT_SHAPE = (96, 96, 3)

# Which converted model to benchmark. Options produced by the training pipeline:
#   mobilenet_vww_i8_w8_a8.fbz     (8-bit)
#   mobilenet_vww_i8_w4_a4_qat.fbz (4-bit QAT)
MODEL_FBZ = os.path.join(MODELS_DIR, 'mobilenet_vww_i8_w8_a8.fbz')

# Clocks (see vww_benchmark.py and brainchip_utils.hardware_utils.AKIDA_CLOCKS_HZ).
MEASURED_CLOCK = AKIDA_CLOCKS_HZ['AKIDA2_FPGA']  # 25 MHz FPGA
PROJECTED_CLOCK = AKIDA_CLOCKS_HZ['AKD2500']     # 1 GHz target (pre-production)

## Load the Akida model

In [2]:
ak_model = akida.Model(MODEL_FBZ)
ak_model.summary()

                Model Summary                 
______________________________________________
Input shape  Output shape  Sequences  Layers
[96, 96, 3]  [1, 1, 2]     1          30    
______________________________________________

______________________________________________________________
Layer (type)                  Output shape  Kernel shape    

========== SW/input_quantizer-dequantizer (Software) =========

input_quantizer (Quantizer)   [96, 96, 3]   N/A             
______________________________________________________________
conv1 (InputConv2D)           [48, 48, 8]   (3, 3, 3, 8)    
______________________________________________________________
conv_dw_1 (DepthwiseConv2D)   [48, 48, 8]   (3, 3, 8, 1)    
______________________________________________________________
conv_pw_1 (Conv2D)            [48, 48, 16]  (1, 1, 8, 16)   
______________________________________________________________
conv_dw_2 (DepthwiseConv2D)   [24, 24, 16]  (3, 3, 16, 1)   
______________________

## Device

`get_akida_device` returns `None` when no compatible hardware is present, in which case the benchmark below is skipped.

In [3]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=ak_model.ip_version)
if device is None:
    print('No compatible Akida hardware device found — benchmark will be skipped.')
else:
    print('Akida device found:', device)

Target Akida device found
Akida device found: <akida.core.HardwareDevice object at 0x7526199f9c30>


## Samples

Akida latency is activity-dependent (it exploits sparsity), so we benchmark on real inputs rather than random data.

In [4]:
from vww_data import get_samples

NUM_SAMPLES = 100
samples = get_samples(DATA_PATH, INPUT_SHAPE, num_samples=NUM_SAMPLES)

Found 98658 images belonging to 2 classes.


## Latency benchmark

Full-model benchmark in both mapping modes, with measured (25 MHz) and projected latency. Cycle count is fixed for a given model + mapping, so `projected_ms = mean_inf_clk / PROJECTED_CLOCK * 1000`.

In [5]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

if device is not None:
    for mm in ['Minimal', 'AllNps']:
        map_mode = getattr(akida.MapMode, mm)
        res = full_model_benchmark(ak_model, device, samples,
                                   map_mode=map_mode, clock_freq=MEASURED_CLOCK)
        projected_ms = res['mean_inf_clk'] / PROJECTED_CLOCK * 1000
        ak_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(ak_model)
        print(f'[{mm}] NPs={num_nps} passes={num_passes} '
              f'latency@25MHz={res["mean_clk_ms"]:.3f} ms '
              f'projected@1000MHz={projected_ms:.3f} ms')
else:
    print('Hardware not available — skipping latency benchmark.')

Power measurement not available: only supported on AKD1500 (IpVersion.v1).
  Repeat 1/10 done.
  Repeat 2/10 done.
  Repeat 3/10 done.
  Repeat 4/10 done.
  Repeat 5/10 done.
  Repeat 6/10 done.
  Repeat 7/10 done.
  Repeat 8/10 done.
  Repeat 9/10 done.
  Repeat 10/10 done.

  Mean inference time:    45.253 ms  (σ=0.334 ms)
  Mean on-chip time:      41.725 ms  (1043134 clocks)
  Total inferences run:   1000
[Minimal] NPs=29 passes=2 latency@25MHz=41.725 ms projected@1000MHz=1.043 ms
Power measurement not available: only supported on AKD1500 (IpVersion.v1).
  Repeat 1/10 done.
  Repeat 2/10 done.
  Repeat 3/10 done.
  Repeat 4/10 done.
  Repeat 5/10 done.
  Repeat 6/10 done.
  Repeat 7/10 done.
  Repeat 8/10 done.
  Repeat 9/10 done.
  Repeat 10/10 done.

  Mean inference time:    36.560 ms  (σ=0.269 ms)
  Mean on-chip time:      34.939 ms  (873473 clocks)
  Total inferences run:   1000
[AllNps] NPs=46 passes=2 latency@25MHz=34.939 ms projected@1000MHz=0.873 ms


## Per-layer benchmark & sparsity

Per-layer latency (Minimal mapping) plus activation sparsity, which drives Akida efficiency.

In [6]:
from brainchip_utils.hardware_utils import per_layer_benchmark
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

if device is not None:
    ak_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    per_layer_results = per_layer_benchmark(ak_model, device, samples,
                                            repeats=NUM_SAMPLES, clock_freq=MEASURED_CLOCK)
    print('Per-layer benchmark complete.')
else:
    # Sparsity can still be computed on the software backend without hardware.
    sparsity_dict = compute_sparsity(ak_model, samples=samples)
    pretty_print_sparsity(sparsity_dict)
    print('Hardware not available — skipped latency; sparsity computed on software backend.')

I0000 00:00:1788455547.287986  869980 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10115 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 SUPER, pci bus id: 0000:04:00.0, compute capability: 8.9



Layer           Sparsity
------------------------
conv1            29.95%
conv_dw_1        37.77%
conv_pw_1        29.72%
conv_dw_2        14.73%
conv_pw_2        18.13%
conv_dw_3        28.79%
conv_pw_3        30.09%
conv_dw_4         5.91%
conv_pw_4         7.15%
conv_dw_5        36.09%
conv_pw_5        28.98%
conv_dw_6        12.94%
conv_pw_6        10.71%
conv_dw_7        38.68%
conv_pw_7        17.33%
conv_dw_8        32.88%
conv_pw_8        25.28%
conv_dw_9        30.55%
conv_pw_9        23.94%
conv_dw_10       35.21%
conv_pw_10       41.61%
conv_dw_11       36.73%
conv_pw_11       39.49%
conv_dw_12       23.00%
conv_pw_12       47.53%
conv_dw_13       37.48%
conv_pw_13       62.47%
predictions       0.00%
------------------------
Mean             27.97%

Layer               Latency (ms)       Clocks
-------------------------------------
input_quantizer           0.0000
conv1                     0.7567
conv_dw_1                 3.3396
conv_pw_1                 2.6263
conv_dw_2  